# 心思维 AI 扒谱助手 · Colab 免费临时运行版

这个笔记本会在你的 Google Colab 会话中启动独立的 MusicMaster 主唱识别服务，并生成一个临时网页链接。打开链接后即可上传主唱音轨、查看进度、预览五线谱并下载结果。

**费用原则：** 本流程不创建 Render 服务、不购买云服务器、不调用付费 API。只使用 Colab 当时提供的免费 CPU/GPU 资源；免费 GPU 不保证每次可用。不要在 Colab 中启用付费计算单元。

**注意：** 上传的音频会传到 Google Colab 临时运行环境；运行时断开后，临时网页和未下载的文件都会消失。请先下载识别结果。当前只处理已经分离好的主唱音轨，不会自动从整首混音中分离人声。

## 使用方法
1. 在 Colab 菜单「运行时 → 更改运行时类型」中，若免费 GPU 可选，可选择 GPU；没有 GPU 时保持 CPU 即可。
2. 点击「运行时 → 全部运行」，等待依赖和 CREPE full 模型准备完成。
3. 最后一格会给出临时网页链接。打开后上传 MP3/WAV 等主唱音轨，点击「开始自动扒谱」。
4. 下载 MIDI、MusicXML、五线谱/简谱源文件和报告。结束后可停止 Colab 运行时。

**首次准备需要下载约 57.7 MB 的 CREPE 模型压缩包及 Python 依赖，耗时取决于 Colab 和网络。**

In [ ]:
# 1. 获取心思维独立验证分支（不会修改你的 GitHub 仓库）
import os, pathlib, subprocess, sys, shutil, re

WORK = pathlib.Path('/content/xinsw-music-transcriber')
BRANCH = 'experiment/musicmaster-standalone'

# 必须是纯 Git HTTPS 地址，不能把 Markdown 链接 [文字](网址) 传给 git。
REPO_URL = 'https://github.com/Trifida/jianpu-gongfang.git'
REPO_URL = re.search(r'https?://[^\s)\]]+', REPO_URL).group(0)

def run_git(args):
    print('执行：git ' + ' '.join(args[:3]) + ' …', flush=True)
    subprocess.run(['git'] + args, check=True)

if (WORK / '.git').is_dir():
    print('发现已有代码目录，正在更新分支……', flush=True)
    run_git(['-C', str(WORK), 'fetch', '--depth', '1', 'origin', BRANCH])
    run_git(['-C', str(WORK), 'checkout', '-B', BRANCH, 'FETCH_HEAD'])
elif WORK.exists():
    # 上次 clone 中断留下的非完整目录，先清理再重新克隆。
    print('发现未完成的旧目录，正在清理……', flush=True)
    shutil.rmtree(WORK)
    run_git(['clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(WORK)])
else:
    run_git(['clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(WORK)])

REPO = WORK / 'services' / 'musicmaster-standalone'
assert (REPO / 'musicmaster' / 'web' / 'static' / 'xinsw.html').is_file(), '心思维测试页面缺失'
print('代码准备完成：', REPO)


In [ ]:
# 2. 分阶段检查并安装依赖：保留 Colab 自带的 NumPy / TensorFlow，显示真实安装进度
import importlib.util
import importlib.metadata as metadata
import subprocess
import sys
import platform
from packaging.requirements import Requirement
from packaging.version import Version

print("运行环境：Python", platform.python_version(), "|", platform.platform())
print("正在检查 Colab 已预装组件……")

def ensure_package(spec):
    req = Requirement(spec)
    dist_name = req.name
    try:
        installed = metadata.version(dist_name)
        okay = (not req.specifier) or req.specifier.contains(Version(installed), prereleases=True)
    except metadata.PackageNotFoundError:
        installed, okay = None, False

    if okay:
        print(f"  [已满足] {dist_name} {installed}")
        return

    print(f"  [开始安装] {spec}（缺失或版本不符合要求）", flush=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "--disable-pip-version-check",
         "install", "--prefer-binary", spec],
        check=True
    )
    print(f"  [安装完成] {spec}", flush=True)

# 不指定 NumPy 版本：避免旧的 numpy<2 约束触发大规模降级/重装。
# 当前 Colab 运行时已预装兼容版本时，直接复用，不重复下载。
groups = [
    ("音频基础库", [
        "scipy>=1.10", "librosa>=0.10", "soundfile>=0.12",
        "pretty_midi>=0.2", "music21>=9.1"
    ]),
    ("乐谱渲染库", [
        "verovio>=4.0", "jianpu-ly>=1.8"
    ]),
    ("网页服务与音高平滑库", [
        "hmmlearn>=0.3", "fastapi>=0.110", "uvicorn>=0.27",
        "python-multipart>=0.0.9", "jsonschema>=4", "httpx>=0.27"
    ])
]

for group_name, specs in groups:
    print(f"\n===== {group_name} =====", flush=True)
    for spec in specs:
        ensure_package(spec)

# CREPE 必须使用 TensorFlow；Colab 通常已预装，只有缺失时才安装。
print("\n===== TensorFlow / CREPE =====", flush=True)
if importlib.util.find_spec("tensorflow") is None:
    ensure_package("tensorflow")
import tensorflow as tf
import numpy as np
print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__)

print("\n正在安装仓库内置的轻量 CREPE 程序（不下载模型权重）……", flush=True)
subprocess.run([
    sys.executable, "-m", "pip", "--disable-pip-version-check",
    "install", "--no-build-isolation", "--no-deps", str(REPO / "vendor" / "crepe")
], check=True)

print("\n正在注册 MusicMaster 本地程序……", flush=True)
subprocess.run([
    sys.executable, "-m", "pip", "--disable-pip-version-check",
    "install", "--no-deps", "-e", str(REPO)
], check=True)

print("\n依赖准备完成。模型将在下一格单独下载并校验；此步骤不会安装人声分离或换音色大型模型。", flush=True)


In [ ]:
# 3. 只下载经过 SHA 校验的 CREPE full 模型（约 57.7 MB），然后启动网页服务
import os, pathlib, subprocess, sys, time, requests
os.environ['PYTHONPATH'] = str(REPO) + os.pathsep + os.environ.get('PYTHONPATH', '')
os.environ['MUSICMASTER_HOST'] = '0.0.0.0'
os.environ['MUSICMASTER_PORT'] = '7860'
model = REPO / 'vendor' / 'crepe' / 'crepe' / 'model-full.h5'
if not model.is_file():
    subprocess.run([sys.executable, str(REPO / 'scripts' / 'prepare_crepe_model.py')], cwd=str(REPO), check=True)
else:
    print('CREPE full 模型已存在，跳过下载。')
log_path = pathlib.Path('/content/xinsw-musicmaster-server.log')
log_file = open(log_path, 'w')
server = subprocess.Popen([sys.executable, '-m', 'uvicorn', 'musicmaster.web.server:app', '--host', '0.0.0.0', '--port', '7860'], cwd=str(REPO), env=os.environ.copy(), stdout=log_file, stderr=subprocess.STDOUT)
healthy = False
for _ in range(60):
    if server.poll() is not None:
        break
    try:
        response = requests.get('http://127.0.0.1:7860/api/health', timeout=2)
        if response.ok and response.json().get('ok'):
            healthy = True
            break
    except Exception:
        time.sleep(1)
if not healthy:
    print(log_path.read_text(errors='replace')[-6000:])
    raise RuntimeError('扒谱服务未能启动，请检查上方错误日志。')
from google.colab import output
page_url = output.eval_js('google.colab.kernel.proxyPort(7860)')
print('服务已启动。请打开下面的心思维测试页面：')
print(page_url.rstrip('/') + '/xinsw.html')
print('注意：保持此 Colab 运行时开启；关闭或断开后，临时页面会失效。')
